# 01 — Data Quality & Cleaning

Challenge Section 1: Exploratory / Data Quality

In [447]:
import numpy as np
import pandas as pd

In [448]:
RAW_PATH = "../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv"

### Explicit dtypes avoid the mixed-type inference warning pandas raises on this file


In [449]:
# Explicit dtypes avoid the mixed-type inference warning pandas raises on this file
DTYPES = {
    "provider_code": "Int64",
    "rider_count": "float64",
    "distance_miles": "float64",
    "rate_class_id": "float64",
    "offline_record_flag": "string",
    "origin_loc_id": "Int64",
    "dest_loc_id": "Int64",
    "fare_settlement_method": "Int64",
    "base_fare": "float64",
    "surcharge_misc": "float64",
    "transit_tax": "float64",
    "driver_tip_payment": "float64",
    "toll_total": "float64",
    "service_improvement_fee": "float64",
    "charge_total": "float64",
    "zone_congestion_fee": "float64",
    "Airport_fee": "float64",
    "congestion_relief_fee": "float64",
}


In [450]:
df_raw = pd.read_csv(
    RAW_PATH,
    dtype=DTYPES,
    parse_dates=["pickup_timestamp", "dropoff_timestamp"],
)

In [451]:
n_raw = len(df_raw)
print(f"Rows: {n_raw:,}  |  Columns: {df_raw.shape[1]}")
df_raw.tail()

Rows: 4,305,006  |  Columns: 20


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee
4305001,2,2025-12-31 23:20:55,2025-12-31 23:24:01,NaN,0.67,NaN,<NA>,216,197,0,7.33,0.0,0.5,0.0,0.0,1.0,8.83,NaN,NaN,0.00
4305002,2,2025-12-31 23:08:47,2025-12-31 23:22:25,NaN,2.95,NaN,<NA>,22,149,0,20.26,0.0,0.5,0.0,0.0,1.0,21.76,NaN,NaN,0.00
4305003,2,2025-12-31 23:29:04,2025-12-31 23:46:32,NaN,9.20,NaN,<NA>,128,50,0,28.45,0.0,0.5,0.0,0.0,1.0,33.20,NaN,NaN,0.75
4305004,2,2025-12-31 23:25:12,2025-12-31 23:29:17,NaN,0.07,NaN,<NA>,114,114,0,31.77,0.0,0.5,0.0,0.0,1.0,36.52,NaN,NaN,0.75
4305005,2,2025-12-31 23:01:11,2025-12-31 23:23:09,NaN,5.50,NaN,<NA>,236,107,0,42.19,0.0,0.5,0.0,0.0,1.0,46.94,NaN,NaN,0.75


In [452]:
df = df_raw

## 2. Overview

In [453]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / n_raw * 100).round(2),
})

,missing_count,missing_pct
rider_count,1195482,27.77
rate_class_id,1195482,27.77
offline_record_flag,1195482,27.77
zone_congestion_fee,1195482,27.77
Airport_fee,1195482,27.77


Observation: `rider_count`, `rate_class_id`, `offline_record_flag`, `zone_congestion_fee`
and `Airport_fee` are all missing for the same ~1.09M rows (29.2%). This lines up with which
`provider_code` supplied the record (some providers, e.g. code 6, do not transmit these
fields) rather than being random — it is a systematic gap, not noise. These columns aren't
part of the five named anomaly checks below, so we address them separately here: impute the
categorical / count fields with the mode and the fee fields with 0 (no fee reported = no fee
charged), keeping an `_missing` flag so downstream models can still use the missingness as a
signal.

In [454]:
df["rider_count_missing"] = df["rider_count"].isna()

df["rider_count"] = df["rider_count"].fillna(df["rider_count"].mode()[0])
df["rate_class_id"] = df["rate_class_id"].fillna(df["rate_class_id"].mode()[0])
df["offline_record_flag"] = df["offline_record_flag"].fillna("N")
df["zone_congestion_fee"] = df["zone_congestion_fee"].fillna(0.0)
df["Airport_fee"] = df["Airport_fee"].fillna(0.0)

df.isna().sum().sum()  # 0 -> no missing values left

np.int64(0)

## 3. Anomaly detection

Each check below reports the count and % of total rows affected, then applies the decided
treatment. Row-level anomaly flags are kept in `drop_mask` / dedicated boolean columns so the
final cleaning step (Section 4) can report the combined impact without double-counting rows
flagged by more than one rule.

### 3.1 Negative fares

In [455]:
flag_negative_fare = df["base_fare"] < 0
n = flag_negative_fare.sum()
print(f"base_fare < 0: {n:,} rows ({n / n_raw * 100:.3f}% of total)")

base_fare < 0: 48,175 rows (1.119% of total)


In [456]:
negative_fares = df[df["charge_total"] < 0]

len(negative_fares)


48869

In [457]:
negative_fares = (df["charge_total"] < 0).sum()

percentage = (negative_fares / len(df)) * 100

print("Negative total charges:", negative_fares)
print(f"Percentage: {percentage:.2f}%")

Negative total charges: 48869
Percentage: 1.14%


In [458]:
df["calculated_charge"] = (
    df["base_fare"]
    + df["surcharge_misc"]
    + df["transit_tax"]
    + df["driver_tip_payment"]
    + df["toll_total"]
    + df["service_improvement_fee"]
    + df["zone_congestion_fee"]
    + df["Airport_fee"]
    + df["congestion_relief_fee"]
)

In [459]:
negative_fares = df["charge_total"] < 0

In [460]:
df.loc[
    negative_fares,
    ["charge_total", "calculated_charge"]
].head()

,charge_total,calculated_charge
46,-7.25,-7.25
127,-61.00,-61.00
161,-7.60,-7.60
265,-13.65,-13.65
364,-8.75,-8.75


In [461]:
df = df[df["charge_total"] >= 0]

In [462]:
print("Negative fares after cleaning:",
      (df["charge_total"] < 0).sum())

Negative fares after cleaning: 0


### 3.2 Trip distance of 0 with a nonzero fare

In [463]:
flag_zero_dist_fare = (df["distance_miles"] == 0) & (df["base_fare"] > 0)
n = flag_zero_dist_fare.sum()
print(f"distance_miles == 0 & base_fare > 0: {n:,} rows ({n / n_raw * 100:.3f}% of total)")

df.loc[flag_zero_dist_fare, "base_fare"].describe()

distance_miles == 0 & base_fare > 0: 148,528 rows (3.450% of total)


count    148528.000000
mean         31.866030
std          29.327256
min           0.010000
25%          15.190000
50%          25.010000
75%          40.000000
max         999.000000
Name: base_fare, dtype: float64

**Decision: drop.** A charged trip with 0 recorded distance means the meter/GPS failed to
log movement — the record can't be trusted for a fare or distance predictor, and there's no
reliable way to impute the missing distance. This is the largest single anomaly (~3.29% of
rows); we drop rather than impute since fabricating distance would bias every distance-based
feature.

In [464]:
df = df[
    ~(
        (df["distance_miles"] == 0) &
        (df["charge_total"] > 0)
    )
]

### 3.3 Passenger count of 0

In [465]:
flag_zero_riders = df["rider_count"] == 0
n = flag_zero_riders.sum()
print(f"rider_count == 0: {n:,} rows ({n / n_raw * 100:.3f}% of total)")


rider_count == 0: 18,144 rows (0.421% of total)


In [466]:
print(df["rider_count"].mode()[0])

1.0


**Decision: impute (mode).** A charged trip implies at least one rider, so 0 is an invalid
entry rather than a real trip we want to lose. `rider_count` is not a primary predictor for
the fare/duration/demand tasks in this challenge, so rather than drop ~0.4% of otherwise-
valid trips we replace 0 with the dataset mode (1 rider) and keep a flag for transparency.


In [467]:
df["rider_count_was_zero"] = flag_zero_riders
df.loc[flag_zero_riders, "rider_count"] = df["rider_count"].mode()[0]

### 3.4 Drop-off at or before pickup

The brief calls out "drop-off before pickup". We check the full non-positive range
(`dropoff <= pickup`), since a trip with 0 duration but a nonzero distance/fare is equally
invalid — the meter clock did not register real elapsed time.

In [468]:
duration_sec = (df["dropoff_timestamp"] - df["pickup_timestamp"]).dt.total_seconds()

flag_bad_duration = duration_sec <= 0
n_before = (duration_sec < 0).sum()
n_zero = (duration_sec == 0).sum()
print(f"dropoff before pickup: {n_before:,} rows ({n_before / n_raw * 100:.3f}%)")
print(f"dropoff == pickup:     {n_zero:,} rows ({n_zero / n_raw * 100:.3f}%)")
print(f"combined <= 0:         {flag_bad_duration.sum():,} rows ({flag_bad_duration.sum() / n_raw * 100:.3f}%)")

dropoff before pickup: 1 rows (0.000%)
dropoff == pickup:     55,993 rows (1.301%)
combined <= 0:         55,994 rows (1.301%)


**Decision: drop.** A non-positive trip duration is impossible for a trip that covered real
distance, and cannot be repaired (we don't know the true pickup/drop-off times). This also
matters directly for Section 2.2 (trip duration prediction), where a duration target of 0 or
negative would corrupt training. Affects ~1.21% of rows.

In [469]:
df = df[
    df["dropoff_timestamp"] >= df["pickup_timestamp"]
]

### 3.5 Unrealistic speed (distance / duration)

In [470]:
duration_hr = duration_sec / 3600
speed_mph = df["distance_miles"] / duration_hr.replace(0, np.nan)

SPEED_LIMIT_MPH = 80  # generous upper bound for city + highway taxi travel

flag_high_speed = (speed_mph > SPEED_LIMIT_MPH).fillna(False)
n = flag_high_speed.sum()
print(f"implied speed > {SPEED_LIMIT_MPH} mph: {n:,} rows ({n / n_raw * 100:.3f}% of total)")

speed_mph[flag_high_speed].sort_values(ascending=False).head(10)

implied speed > 80 mph: 1,127 rows (0.026% of total)


3244705    3.748310e+06
4021642    2.764939e+06
3295859    1.942734e+06
3834657    1.829997e+06
4275899    1.575650e+06
4024102    1.532415e+06
3368119    1.372315e+06
3019764    1.226017e+06
3694539    1.208921e+06
3747997    1.150280e+06
dtype: float64

**Decision: filter (drop).** Implied speeds in the hundreds of thousands of mph trace back
to a handful of `distance_miles` values in the hundreds of thousands (clear unit/sensor
errors) — not something to impute. A 100 mph cap is generous for taxi travel (including
highway runs to JFK/Newark/Nassau/Westchester) and only removes ~0.02% of rows.

In [471]:
n_before = len(df)

# Drop rows where implied speed exceeds 80 mph (keep NaN-speed rows)
df = df.loc[~flag_high_speed].copy()

n_dropped = n_before - len(df)
print(f"dropped {n_dropped:,} rows ({n_dropped / n_before * 100:.3f}% of total); {len(df):,} rows remain")

dropped 1,127 rows (0.027% of total); 4,106,224 rows remain


## Save Preprocessed Data


In [472]:
path = "../data/processed"  # change to your folder

import os
os.makedirs(path, exist_ok=True)

df.to_csv(os.path.join(path, "df_cleaned_12.csv"), index=False)
print(f"saved {len(df):,} rows to {os.path.join(path, 'df_cleaned.csv')}")

saved 4,106,224 rows to ../data/processed/df_cleaned.csv
